# Computational Physics Final Project  
## 3D Projectile Motion with and without Air Resistance

This notebook simulates a 3D ball model performing projectile motion from an initial height.

Selectable/adjustable conditions:

- `motion_type`: horizontal projectile or oblique projectile
- `drag_enabled`: no resistance or air resistance
- `initial_height`
- `initial_speed`
- `launch_angle_deg`
- `azimuth_angle_deg`
- ball radius, mass, drag coefficient, air density, time step, and simulation time

The numerical method used here is the fourth-order Runge-Kutta method (RK4).

## 1. Physics Model

The position vector is:

\[
\vec r = (x, y, z)
\]

The velocity vector is:

\[
\vec v = (v_x, v_y, v_z)
\]

Without air resistance:

\[
\frac{d\vec v}{dt} = (0,0,-g)
\]

With quadratic air resistance:

\[
\vec F_d = -\frac{1}{2}\rho C_d A |\vec v|\vec v
\]

\[
\frac{d\vec v}{dt} = (0,0,-g) - \frac{1}{2m}\rho C_d A |\vec v|\vec v
\]

where \(A = \pi r^2\) is the cross-sectional area of the ball.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
def initial_velocity(initial_speed, launch_angle_deg=45.0, azimuth_angle_deg=0.0, motion_type="oblique"):
    """
    Calculate the initial 3D velocity components.

    motion_type:
    - "horizontal": vertical initial velocity is zero
    - "oblique": vertical initial velocity depends on launch_angle_deg

    launch_angle_deg:
    - elevation angle above the horizontal plane

    azimuth_angle_deg:
    - horizontal direction angle in the x-y plane
    """
    phi = np.radians(azimuth_angle_deg)

    if motion_type == "horizontal":
        vx = initial_speed * np.cos(phi)
        vy = initial_speed * np.sin(phi)
        vz = 0.0
    else:
        theta = np.radians(launch_angle_deg)
        horizontal_speed = initial_speed * np.cos(theta)
        vx = horizontal_speed * np.cos(phi)
        vy = horizontal_speed * np.sin(phi)
        vz = initial_speed * np.sin(theta)

    return np.array([vx, vy, vz], dtype=float)


def projectile_derivatives(state, params):
    """
    state = [x, y, z, vx, vy, vz]
    returns dstate/dt
    """
    x, y, z, vx, vy, vz = state

    g = params["g"]
    mass = params["mass"]
    radius = params["radius"]
    rho = params["air_density"]
    Cd = params["drag_coefficient"]
    drag_enabled = params["drag_enabled"]

    velocity = np.array([vx, vy, vz], dtype=float)
    speed = np.linalg.norm(velocity)

    acceleration = np.array([0.0, 0.0, -g], dtype=float)

    if drag_enabled and speed > 0:
        area = np.pi * radius**2
        drag_acceleration = -(0.5 * rho * Cd * area / mass) * speed * velocity
        acceleration += drag_acceleration

    return np.array([vx, vy, vz, acceleration[0], acceleration[1], acceleration[2]], dtype=float)


def rk4_step(state, dt, params):
    """
    One fourth-order Runge-Kutta step.
    """
    k1 = projectile_derivatives(state, params)
    k2 = projectile_derivatives(state + 0.5 * dt * k1, params)
    k3 = projectile_derivatives(state + 0.5 * dt * k2, params)
    k4 = projectile_derivatives(state + dt * k3, params)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def simulate_projectile(
    motion_type="oblique",
    drag_enabled=False,
    initial_height=10.0,
    initial_speed=20.0,
    launch_angle_deg=45.0,
    azimuth_angle_deg=0.0,
    mass=0.145,
    radius=0.0366,
    drag_coefficient=0.47,
    air_density=1.225,
    g=9.81,
    dt=0.01,
    max_time=10.0,
):
    """
    Simulate 3D projectile motion until the ball reaches the ground or max_time is reached.
    """
    params = {
        "motion_type": motion_type,
        "drag_enabled": drag_enabled,
        "initial_height": initial_height,
        "initial_speed": initial_speed,
        "launch_angle_deg": launch_angle_deg,
        "azimuth_angle_deg": azimuth_angle_deg,
        "mass": mass,
        "radius": radius,
        "drag_coefficient": drag_coefficient,
        "air_density": air_density,
        "g": g,
        "dt": dt,
        "max_time": max_time,
    }

    r0 = np.array([0.0, 0.0, initial_height], dtype=float)
    v0 = initial_velocity(initial_speed, launch_angle_deg, azimuth_angle_deg, motion_type)
    state = np.concatenate([r0, v0])

    times = [0.0]
    states = [state.copy()]

    t = 0.0
    while t < max_time:
        new_state = rk4_step(state, dt, params)
        t += dt

        if new_state[2] < 0:
            previous_state = state
            alpha = previous_state[2] / (previous_state[2] - new_state[2])
            impact_state = previous_state + alpha * (new_state - previous_state)
            impact_time = (t - dt) + alpha * dt
            states.append(impact_state.copy())
            times.append(impact_time)
            break

        states.append(new_state.copy())
        times.append(t)
        state = new_state

    return np.array(times), np.array(states), params

In [ ]:
def summarize_simulation(times, states):
    final_state = states[-1]
    x, y, z, vx, vy, vz = final_state

    horizontal_range = np.sqrt(x**2 + y**2)
    max_height = np.max(states[:, 2])
    flight_time = times[-1]
    final_speed = np.linalg.norm(final_state[3:6])

    return {
        "flight_time_s": flight_time,
        "range_m": horizontal_range,
        "max_height_m": max_height,
        "final_speed_m_per_s": final_speed,
        "final_x_m": x,
        "final_y_m": y,
    }


def print_summary(title, times, states):
    result = summarize_simulation(times, states)
    print(title)
    print("-" * len(title))
    for key, value in result.items():
        print(f"{key}: {value:.4f}")


def plot_trajectory_3d(times, states, title="3D Projectile Motion"):
    x, y, z = states[:, 0], states[:, 1], states[:, 2]

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(x, y, z, label="trajectory")
    ax.scatter([x[0]], [y[0]], [z[0]], s=50, label="start")
    ax.scatter([x[-1]], [y[-1]], [z[-1]], s=50, label="impact")

    ax.set_title(title)
    ax.set_xlabel("x position (m)")
    ax.set_ylabel("y position (m)")
    ax.set_zlabel("z height (m)")
    ax.legend()
    plt.show()


def plot_motion_graphs(times, states, title_prefix="Projectile Motion"):
    x, y, z = states[:, 0], states[:, 1], states[:, 2]
    vx, vy, vz = states[:, 3], states[:, 4], states[:, 5]
    speed = np.sqrt(vx**2 + vy**2 + vz**2)

    plt.figure()
    plt.plot(times, z)
    plt.title(f"{title_prefix}: Height vs Time")
    plt.xlabel("time (s)")
    plt.ylabel("height z (m)")
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(times, speed)
    plt.title(f"{title_prefix}: Speed vs Time")
    plt.xlabel("time (s)")
    plt.ylabel("speed (m/s)")
    plt.grid(True)
    plt.show()

    horizontal_distance = np.sqrt(x**2 + y**2)
    plt.figure()
    plt.plot(horizontal_distance, z)
    plt.title(f"{title_prefix}: Height vs Horizontal Distance")
    plt.xlabel("horizontal distance (m)")
    plt.ylabel("height z (m)")
    plt.grid(True)
    plt.show()

In [ ]:
def compare_drag_and_no_drag(
    motion_type="oblique",
    initial_height=10.0,
    initial_speed=20.0,
    launch_angle_deg=45.0,
    azimuth_angle_deg=0.0,
    mass=0.145,
    radius=0.0366,
    drag_coefficient=0.47,
    air_density=1.225,
    g=9.81,
    dt=0.01,
    max_time=10.0,
):
    t_no, s_no, _ = simulate_projectile(
        motion_type=motion_type,
        drag_enabled=False,
        initial_height=initial_height,
        initial_speed=initial_speed,
        launch_angle_deg=launch_angle_deg,
        azimuth_angle_deg=azimuth_angle_deg,
        mass=mass,
        radius=radius,
        drag_coefficient=drag_coefficient,
        air_density=air_density,
        g=g,
        dt=dt,
        max_time=max_time,
    )

    t_drag, s_drag, _ = simulate_projectile(
        motion_type=motion_type,
        drag_enabled=True,
        initial_height=initial_height,
        initial_speed=initial_speed,
        launch_angle_deg=launch_angle_deg,
        azimuth_angle_deg=azimuth_angle_deg,
        mass=mass,
        radius=radius,
        drag_coefficient=drag_coefficient,
        air_density=air_density,
        g=g,
        dt=dt,
        max_time=max_time,
    )

    print_summary("Without air resistance", t_no, s_no)
    print()
    print_summary("With air resistance", t_drag, s_drag)

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot(s_no[:, 0], s_no[:, 1], s_no[:, 2], label="without air resistance")
    ax.plot(s_drag[:, 0], s_drag[:, 1], s_drag[:, 2], label="with air resistance")

    ax.set_title("3D Trajectory Comparison")
    ax.set_xlabel("x position (m)")
    ax.set_ylabel("y position (m)")
    ax.set_zlabel("z height (m)")
    ax.legend()
    plt.show()

    return (t_no, s_no), (t_drag, s_drag)

In [ ]:
def animate_projectile_3d(times, states, title="3D Ball Projectile Animation", interval=30, trail=True):
    x, y, z = states[:, 0], states[:, 1], states[:, 2]

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.set_xlim(np.min(x) - 1, np.max(x) + 1)
    ax.set_ylim(np.min(y) - 1, np.max(y) + 1)
    ax.set_zlim(0, max(np.max(z) + 1, 1))

    ax.set_title(title)
    ax.set_xlabel("x position (m)")
    ax.set_ylabel("y position (m)")
    ax.set_zlabel("z height (m)")

    ball, = ax.plot([], [], [], marker="o", markersize=10, linestyle="")
    path, = ax.plot([], [], [], linewidth=1)

    def init():
        ball.set_data([], [])
        ball.set_3d_properties([])
        path.set_data([], [])
        path.set_3d_properties([])
        return ball, path

    def update(frame):
        ball.set_data([x[frame]], [y[frame]])
        ball.set_3d_properties([z[frame]])

        if trail:
            path.set_data(x[:frame+1], y[:frame+1])
            path.set_3d_properties(z[:frame+1])

        return ball, path

    anim = FuncAnimation(
        fig,
        update,
        frames=len(times),
        init_func=init,
        interval=interval,
        blit=False,
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())

## 2. Default Simulation

The following cell runs the default case. You can change the values directly in the function call.

In [ ]:
times, states, params = simulate_projectile(
    motion_type="oblique",
    drag_enabled=True,
    initial_height=10.0,
    initial_speed=20.0,
    launch_angle_deg=45.0,
    azimuth_angle_deg=30.0,
    mass=0.145,
    radius=0.0366,
    drag_coefficient=0.47,
    air_density=1.225,
    dt=0.01,
    max_time=10.0,
)

print_summary("Default simulation: oblique motion with air resistance", times, states)
plot_trajectory_3d(times, states, "3D Ball Trajectory with Air Resistance")
plot_motion_graphs(times, states, "With Air Resistance")

## 3. Comparison: With Resistance vs Without Resistance

This comparison is useful for the final presentation because it clearly shows the physical effect of air resistance.

In [ ]:
(no_drag_data, drag_data) = compare_drag_and_no_drag(
    motion_type="oblique",
    initial_height=10.0,
    initial_speed=20.0,
    launch_angle_deg=45.0,
    azimuth_angle_deg=30.0,
    mass=0.145,
    radius=0.0366,
    drag_coefficient=0.47,
    air_density=1.225,
    dt=0.01,
    max_time=10.0,
)

## 4. 3D Animation

Run this cell to display the ball animation. For large simulations, increase `dt` or decrease `max_time` to make the animation lighter.

In [ ]:
display(animate_projectile_3d(drag_data[0], drag_data[1], "3D Ball Motion with Air Resistance"))

## 5. Interactive Control Panel

If `ipywidgets` is installed, run the following cell and adjust the sliders/drop-down menus.  
This allows the user to choose horizontal/oblique motion, enable/disable drag, and adjust initial values.

In [ ]:
def run_interactive_simulation(
    motion_type,
    drag_enabled,
    initial_height,
    initial_speed,
    launch_angle_deg,
    azimuth_angle_deg,
    mass,
    radius,
    drag_coefficient,
    air_density,
    dt,
    max_time,
):
    times, states, _ = simulate_projectile(
        motion_type=motion_type,
        drag_enabled=drag_enabled,
        initial_height=initial_height,
        initial_speed=initial_speed,
        launch_angle_deg=launch_angle_deg,
        azimuth_angle_deg=azimuth_angle_deg,
        mass=mass,
        radius=radius,
        drag_coefficient=drag_coefficient,
        air_density=air_density,
        dt=dt,
        max_time=max_time,
    )

    title = f"{motion_type.title()} Motion | Drag: {drag_enabled}"
    print_summary(title, times, states)
    plot_trajectory_3d(times, states, title)
    plot_motion_graphs(times, states, title)


if WIDGETS_AVAILABLE:
    interact(
        run_interactive_simulation,
        motion_type=widgets.Dropdown(options=["horizontal", "oblique"], value="oblique", description="Motion type"),
        drag_enabled=widgets.Checkbox(value=True, description="Air resistance"),
        initial_height=widgets.FloatSlider(value=10.0, min=0.0, max=100.0, step=1.0, description="Height (m)"),
        initial_speed=widgets.FloatSlider(value=20.0, min=1.0, max=100.0, step=1.0, description="Speed (m/s)"),
        launch_angle_deg=widgets.FloatSlider(value=45.0, min=0.0, max=89.0, step=1.0, description="Launch angle"),
        azimuth_angle_deg=widgets.FloatSlider(value=30.0, min=0.0, max=360.0, step=5.0, description="Horizontal angle"),
        mass=widgets.FloatSlider(value=0.145, min=0.01, max=5.0, step=0.01, description="Mass (kg)"),
        radius=widgets.FloatSlider(value=0.0366, min=0.005, max=0.3, step=0.001, description="Radius (m)"),
        drag_coefficient=widgets.FloatSlider(value=0.47, min=0.0, max=2.0, step=0.01, description="Cd"),
        air_density=widgets.FloatSlider(value=1.225, min=0.0, max=2.0, step=0.025, description="Air density"),
        dt=widgets.FloatSlider(value=0.01, min=0.005, max=0.1, step=0.005, description="dt (s)"),
        max_time=widgets.FloatSlider(value=10.0, min=1.0, max=30.0, step=1.0, description="Max time"),
    )
else:
    print("ipywidgets is not installed. Change the values manually in the normal simulation cells.")

## 6. Error Analysis

For numerical error analysis, the same physical condition is simulated using different time steps.  
If the result changes very little when `dt` becomes smaller, the numerical method is stable enough for this project.

In [ ]:
def timestep_error_analysis():
    dt_values = [0.1, 0.05, 0.02, 0.01, 0.005]
    results = []

    for dt in dt_values:
        times, states, _ = simulate_projectile(
            motion_type="oblique",
            drag_enabled=True,
            initial_height=10.0,
            initial_speed=20.0,
            launch_angle_deg=45.0,
            azimuth_angle_deg=30.0,
            dt=dt,
            max_time=10.0,
        )
        summary = summarize_simulation(times, states)
        results.append((dt, summary["flight_time_s"], summary["range_m"], summary["max_height_m"]))

    print("dt        flight_time(s)    range(m)    max_height(m)")
    print("-" * 55)
    for row in results:
        print(f"{row[0]:<9.4f} {row[1]:<15.6f} {row[2]:<11.6f} {row[3]:.6f}")

    dt_values = [r[0] for r in results]
    ranges = [r[2] for r in results]

    plt.figure()
    plt.plot(dt_values, ranges, marker="o")
    plt.gca().invert_xaxis()
    plt.title("Time Step Error Analysis")
    plt.xlabel("time step dt (s)")
    plt.ylabel("computed range (m)")
    plt.grid(True)
    plt.show()

timestep_error_analysis()

## 7. Boundary Cases

Useful boundary cases for the report:

1. Initial height = 0: the ball starts from the ground.  
2. Launch angle = 0: this becomes horizontal projectile motion.  
3. Drag coefficient = 0: this becomes the same as the no-resistance case.  
4. Very large drag coefficient: the projectile loses speed quickly and the range becomes much shorter.  
5. Very small time step: the result should become more stable, but the computation becomes slower.